In [ ]:
MODEL_PATH         = "/content/drive/MyDrive/efficientnet_model/efficientnet.keras"
NGROK_AUTHTOKEN    = ""  # Thay bằng token thật của bạn
PORT               = 8000
IMG_SIZE           = 300
BATCH_SIZE         = 32
LABELS             = ["nsfw", "safe", "violence"]
NSFW_THRESHOLD     = 0.6
VIOLENCE_THRESHOLD = 0.6
# ══════════════════════════════════════════════════════════════════════════════

import subprocess
import sys
import os
import io
import logging
import asyncio
import time
from typing import Annotated

# 1. CÀI ĐẶT THƯ VIỆN CẬP NHẬT (Không ép hạ cấp Numpy)
def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + list(pkgs))

print("📦 Đang cập nhật các thư viện lên phiên bản mới nhất...")
_pip(
    "fastapi",
    "uvicorn[standard]",
    "python-multipart",
    "Pillow",
    "pyngrok",
    "nest_asyncio",
)
print("✅ Cài đặt hoàn tất!")

# Mount Google Drive
from google.colab import drive
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

import numpy as np
import tensorflow as tf
import nest_asyncio
from PIL import Image
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import uvicorn
from pyngrok import ngrok, conf

nest_asyncio.apply()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
)
logger = logging.getLogger("image_moderation")

# --- Pydantic Models ---
class FramePrediction(BaseModel):
    frame:      str
    label:      str
    confidence: float = Field(..., ge=0.0, le=1.0)
    scores:     dict[str, float]

class BatchPredictResponse(BaseModel):
    total:       int
    predictions: list[FramePrediction]

class HealthResponse(BaseModel):
    status:       str
    model_loaded: bool
    labels:       list[str]
    gpu_available: bool

_model = None
_gpu_available = False

# 2. LOAD MODEL TỐI ƯU CHO KERAS 3 (TensorFlow 2.17+)
def load_model():
    global _model, _gpu_available
    if not os.path.isfile(MODEL_PATH):
        raise FileNotFoundError(
            f"Không tìm thấy model: {MODEL_PATH}\nHãy kiểm tra lại đường dẫn trên Google Drive."
        )
    print(f"🖼️  Loading EfficientNet from: {MODEL_PATH}")

    gpus = tf.config.list_physical_devices("GPU")
    _gpu_available = len(gpus) > 0

    # Sử dụng compile=False vì ta chỉ dùng model để Predict, tránh lỗi không tương thích config lớp của Keras cũ/mới
    _model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    print(f"✅ Model loaded — GPU available: {_gpu_available} ({len(gpus)} device(s))")
    return _model

# 3. TIỀN XỬ LÝ ẢNH AN TOÀN VỚI PILLOW + TF TỐI ƯU TỐC ĐỘ
def preprocess_image(image_bytes: bytes) -> np.ndarray:
    # Đọc ảnh bằng Pillow trước giúp tránh lỗi định dạng lạ từ luồng byte trực tiếp của FastAPI
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    img = img.resize((IMG_SIZE, IMG_SIZE))
    img_array = np.array(img, dtype=np.float32)
    return img_array


def predict_batch(images: list[np.ndarray]) -> list[dict]:
    batch = np.stack(images, axis=0)

    # Thực hiện inference
    raw_preds = _model(batch, training=False).numpy()

    results = []
    for pred in raw_preds:
        prob_map = {label: float(pred[i]) for i, label in enumerate(LABELS)}

        if prob_map.get("nsfw", 0.0) > NSFW_THRESHOLD:
            predicted_label = "nsfw"
        elif prob_map.get("violence", 0.0) > VIOLENCE_THRESHOLD:
            predicted_label = "violence"
        else:
            predicted_label = "safe"

        results.append({
            "label":      predicted_label,
            "confidence": prob_map[predicted_label],
            "scores":     prob_map,
        })
    return results

print("\n🚀 Đang khởi tạo model...")
load_model()
print("✅ Model đã sẵn sàng!\n")

# --- FastAPI App Setup ---
app = FastAPI(
    title="VideoGuard Image Moderation",
    description="EfficientNet: phân loại frame video thành nsfw / safe / violence.",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["GET", "POST"],
    allow_headers=["*"],
)

@app.get("/health", response_model=HealthResponse, tags=["health"])
def health():
    return HealthResponse(
        status="ok",
        model_loaded=_model is not None,
        labels=LABELS,
        gpu_available=_gpu_available,
    )

async def _read_and_preprocess(upload: UploadFile) -> tuple[str, np.ndarray | None]:
    filename = upload.filename or f"unknown_{id(upload)}.jpg"
    try:
        raw = await upload.read()
        # Chạy tiền xử lý đồng bộ trong ThreadPool để không block Event Loop của FastAPI
        loop = asyncio.get_running_loop()
        arr = await loop.run_in_executor(None, preprocess_image, raw)
        return filename, arr
    except Exception as exc:
        logger.warning(f"Thất bại khi giải mã frame {filename}: {exc}")
        return filename, None

@app.post(
    "/images/predict/batch",
    response_model=BatchPredictResponse,
    tags=["predict"],
    summary="Predict labels for a batch of frames",
)
async def predict_frames_batch(
    files: Annotated[list[UploadFile], File(description="Danh sách frame JPEG/PNG")],
):
    if not files:
        raise HTTPException(status_code=422, detail="Không có file nào được upload.")
    if len(files) > BATCH_SIZE:
        raise HTTPException(
            status_code=422,
            detail=f"Vượt quá số lượng frame cho phép. Tối đa: {BATCH_SIZE}.",
        )

    tasks = [_read_and_preprocess(upload) for upload in files]
    results = await asyncio.gather(*tasks)

    images, filenames, failed_files = [], [], []
    for filename, arr in results:
        if arr is not None:
            images.append(arr)
            filenames.append(filename)
        else:
            failed_files.append(filename)

    if not images:
        raise HTTPException(
            status_code=422,
            detail=f"Không thể giải mã bất kỳ file nào trong tổng số {len(files)} ảnh.",
        )
    if failed_files:
        logger.warning(f"Bỏ qua {len(failed_files)} file lỗi: {', '.join(failed_files)}")

    logger.info(f"[predict] Đang chạy inference trên {len(images)} frames")

    # Gọi hàm xử lý dự đoán
    raw_results = predict_batch(images)

    predictions = [
        FramePrediction(frame=fname, **res)
        for fname, res in zip(filenames, raw_results)
    ]
    return BatchPredictResponse(total=len(predictions), predictions=predictions)


# --- Khởi chạy Server và kết nối Ngrok ---
if not NGROK_AUTHTOKEN or NGROK_AUTHTOKEN == "3":
    raise ValueError("❌ Hãy điền NGROK_AUTHTOKEN chính xác từ tài khoản dashboard.ngrok.com của bạn!")

conf.get_default().auth_token = NGROK_AUTHTOKEN
public_url = None

try:
    ngrok.kill()
    public_url = ngrok.connect(PORT, "http").public_url
    print(f"\n{'='*60}")
    print(f"🌐 Public API URL  : {public_url}")
    print(f"📖 Swagger Docs    : {public_url}/docs")
    print(f"❤️  Health Check   : {public_url}/health")
    print(f"🖼️  Predict Batch  : POST {public_url}/images/predict/batch")
    print(f"\n💡 Copy cấu hình này vào file (.env) của Go backend:")
    print(f"   AI_FRAME_MODERATOR_URL={public_url}")
    print(f"{'='*60}\n")

    # Cấu hình uvicorn
    config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info")
    server = uvicorn.Server(config)
    
    # Ép uvicorn chạy đồng bộ trên Event Loop hiện tại của Colab (Blocking mode)
    loop = asyncio.get_event_loop()
    loop.run_until_complete(server.serve())

except KeyboardInterrupt:
    print("\n🛑 Bạn đã chủ động dừng Server bằng nút Stop trên Colab.")
except Exception as e:
    print(f"❗️ Server dừng lại do có lỗi xảy ra: {e}")

finally:
    # Khối này luôn chạy khi bạn bấm nút Stop (Hình ô vuông)
    if public_url:
        print("🔌 Đang ngắt kết nối ngrok tunnel...")
        try:
            ngrok.disconnect(public_url)
        except Exception:
            pass
    ngrok.kill()
    print("✅ Server và ngrok đã được dọn dẹp sạch sẽ.")
